<div style="padding:10px 0; text-align:left; color:#22396F;"><h1>RNN & LSTM - Medical Condition Classification</h1></div>
<p>Deep Learning on the Medical Condition Assistant symptom dataset</p>

<div style="border-top:2px solid gray; border-bottom:2px solid gray; padding:0 0 7px; text-align:left;"><h3>Introduction</h3></div>

#### Goal
This notebook trains RNN and LSTM sequence classifiers on the project’s `dataset/usable_dataset.csv`. A symptom description is tokenised, encoded as a sequence, then mapped to one condition label.

The app remains retrieval-grounded: the deep model ranks labels, while symptoms, causes, warnings and recommendations displayed in Streamlit are always loaded from the matching local RAG record.

#### Important data limitation
The base dataset contains one NHS document per condition. To create several inputs per condition, this notebook derives overlapping token chunks from the original `Symptoms` text. A held-out chunk can show engineering training behaviour but is **not** independent patient validation or medical diagnostic accuracy.

<div style="border-top:2px solid gray; border-bottom:2px solid gray; padding:0 0 7px; text-align:left;"><h3>Mathematical Foundation</h3></div>

#### RNN
For each symptom token embedding $x_t$, the RNN updates its hidden memory:

$$h_t = \tanh(W_xx_t + W_hh_{t-1} + b)$$

The final hidden state is classified with softmax:

$$p(y\mid x) = \mathrm{softmax}(W_oh_T+b_o)$$

#### LSTM
LSTM adds a cell state and gates to retain or discard sequence information:

$$f_t = \sigma(W_f[x_t,h_{t-1}] + b_f), \quad i_t = \sigma(W_i[x_t,h_{t-1}] + b_i)$$
$$c_t = f_t c_{t-1} + i_t\tanh(W_c[x_t,h_{t-1}] + b_c)$$

Both project models use: `Embedding → RNN/LSTM → Dropout → Dense(condition labels)`.

<div style="border-top:2px solid gray; border-bottom:2px solid gray; padding:0 0 7px; text-align:left;"><h3>Project Workflow</h3></div>
1. Load the project symptom dataset.
2. Tokenise symptoms and make overlapping sequences.
3. Split chunks by condition into train and held-out chunk sets.
4. Build the RNN or LSTM layers.
5. Train each epoch and save a checkpoint.
6. Keep `best.pt`, then use it from Flask and Streamlit.

### 1. Import required libraries

In [ ]:
from pathlib import Path
import sys, json, argparse
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'model_API').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_API.deep_learning_pipeline import (
    CHECKPOINTS_DIR, DATASET_PATH, build_chunks, build_vocabulary, create_dataset,
    create_model, load_condition_documents, split_chunks_by_condition, train,
)
from model_API.deep_learning_inference import predict

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_PATH}')
print(f'Checkpoint root: {CHECKPOINTS_DIR}')

### 2. Load the base dataset

In [ ]:
documents = load_condition_documents(DATASET_PATH)
summary = pd.DataFrame(
    {'Condition': [label for label, _ in documents], 'Token count': [len(tokens) for _, tokens in documents]}
)
print(f'Condition documents: {len(documents)}')
display(summary.head())
summary['Token count'].describe()

### 3. Create symptom sequences
Each long symptom document is split into overlapping 96-token windows. This gives the sequence models several source-derived symptom inputs while keeping every token inside the project dataset.

In [ ]:
CHUNK_SIZE = 96
CHUNK_STRIDE = 72
MAX_CHUNKS_PER_CONDITION = 4
MAX_SEQUENCE_LENGTH = 96

chunks = build_chunks(
    documents, chunk_size=CHUNK_SIZE, stride=CHUNK_STRIDE,
    max_chunks_per_condition=MAX_CHUNKS_PER_CONDITION,
)
print(f'Symptom chunks: {len(chunks)}')
print('First chunk label:', chunks[0].label)
print('First chunk tokens:', ' '.join(chunks[0].tokens[:24]))

### 4. Split into training and held-out chunks
The split keeps at least one source chunk in training for every condition. The validation set therefore evaluates within-document chunk generalisation only.

In [ ]:
train_chunks, validation_chunks = split_chunks_by_condition(chunks, validation_fraction=0.25, seed=42)
labels = sorted({label for label, _ in documents})
label_to_id = {label: index for index, label in enumerate(labels)}
vocabulary = build_vocabulary(train_chunks, minimum_frequency=2)
train_dataset = create_dataset(train_chunks, vocabulary, label_to_id, MAX_SEQUENCE_LENGTH)
print(f'Train chunks: {len(train_chunks)}')
print(f'Held-out chunks: {len(validation_chunks)}')
print(f'Vocabulary size: {len(vocabulary)}')
print(f'One model input shape: {tuple(train_dataset[0]["input_ids"].shape)}')

### 5. Build the RNN and LSTM layers
The cells below preview the exact PyTorch architectures used by the training function.

In [ ]:
rnn_preview = create_model('rnn', len(vocabulary), len(labels), embedding_dim=96, hidden_dim=128)
lstm_preview = create_model('lstm', len(vocabulary), len(labels), embedding_dim=96, hidden_dim=128)
print('RNN architecture:\n', rnn_preview)
print('\nLSTM architecture:\n', lstm_preview)

### 6. Configure training
Set `MODEL_TO_TRAIN` to `rnn`, `lstm`, or `all`. Every epoch is saved under `model_API/checkpoints/deep_learning/<model>/epoch_XXX.pt`; the highest held-out chunk Top-1 checkpoint is also saved as `best.pt`.

In [ ]:
MODEL_TO_TRAIN = 'all'  # 'rnn', 'lstm', or 'all'
EPOCHS = 8
BATCH_SIZE = 32
DEVICE = 'auto'  # auto, cpu, or cuda

args = argparse.Namespace(
    data=str(DATASET_PATH), epochs=EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=1e-3, weight_decay=1e-4, embedding_dim=96, hidden_dim=128,
    max_length=MAX_SEQUENCE_LENGTH, chunk_size=CHUNK_SIZE, chunk_stride=CHUNK_STRIDE,
    max_chunks_per_condition=MAX_CHUNKS_PER_CONDITION, minimum_token_frequency=2,
    validation_fraction=0.25, seed=42, device=DEVICE,
)
args

### 7. Train and save checkpoints

In [ ]:
models_to_train = ('rnn', 'lstm') if MODEL_TO_TRAIN == 'all' else (MODEL_TO_TRAIN,)
training_results = []
for model_kind in models_to_train:
    print(f'\nTraining {model_kind.upper()}...')
    result = train(model_kind, args)
    training_results.append(result)
    print(json.dumps(result, indent=2))

print('Checkpoints saved to:', CHECKPOINTS_DIR)
print('Restart the Flask API after training so it loads the new best.pt files.')

### 8. Evaluate checkpoint results
The checkpoint metadata stores training loss and held-out chunk Top-1/Top-3 values per epoch. These metrics track the experiment only; they do not establish medical diagnostic performance.

In [ ]:
for model_kind in ('rnn', 'lstm'):
    metadata_path = CHECKPOINTS_DIR / model_kind / 'training_metadata.json'
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        print(f'\n{model_kind.upper()} best checkpoint:', metadata['checkpoint'])
        display(pd.DataFrame(metadata['history']))
    else:
        print(f'{model_kind.upper()} has not been trained yet.')

### 9. Make a prediction
After training, this cell loads `best.pt`. The Streamlit options **Deep Learning (RNN)** and **Deep Learning (LSTM)** call the same checkpoint path through Flask.

In [ ]:
MODEL_FOR_PREDICTION = 'lstm'
SYMPTOMS = 'wheezing, chest tightness, coughing and shortness of breath at night'

try:
    prediction = predict(MODEL_FOR_PREDICTION, SYMPTOMS, top_k=3)
    print(json.dumps(prediction, indent=2))
except Exception as error:
    print(error)